# 05 · GraphSAGE — Inductive Node Classification

## What this notebook covers
GraphSAGE (Hamilton et al. 2017) was a turning point: it made GNNs *inductive*. You can train on a graph and embed nodes that weren't in the training set — critical for production systems where the graph grows continuously.

## Why inductiveness matters
GCN and spectral methods compute embeddings that are tied to the fixed adjacency matrix. If a new node appears, you need to retrain or at least re-compute global structures. GraphSAGE instead learns an *aggregator function* that can be applied to any neighbourhood, regardless of whether those nodes were seen during training.

**GraphSAGE aggregators** (all three are implemented in PyTorch Geometric):
- **Mean**: average of neighbour embeddings + current embedding
- **LSTM**: feed a random permutation of neighbours into an LSTM (not permutation-invariant by design; works well in practice)
- **Max-pool**: element-wise max after a MLP transform

## Neighbourhood sampling
For large graphs, computing the full neighbourhood is too expensive. GraphSAGE introduced mini-batch training with neighbourhood sampling: at each layer, sample a fixed number of neighbours rather than using all of them. This is now the standard approach for large-scale graph training (used in Pinterest PinSage, LinkedIn's economic graph embedding, etc.).

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.datasets import Planetoid
    from torch_geometric.nn import SAGEConv
    from torch_geometric.loader import NeighborLoader
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

import torch.nn as nn
import networkx as nx

In [ ]:
if HAS_PYG:
    dataset = Planetoid(root="/tmp/Cora", name="Cora")
    data = dataset[0]

    class GraphSAGE(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")
            self.conv2 = SAGEConv(hidden_channels, out_channels, aggr="mean")

        def forward(self, x, edge_index):
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=0.5, training=self.training)
            return self.conv2(x, edge_index)

    sage = GraphSAGE(dataset.num_features, 64, dataset.num_classes)
    opt  = torch.optim.Adam(sage.parameters(), lr=0.01)

    best_val, best_test = 0, 0
    for epoch in range(200):
        sage.train()
        opt.zero_grad()
        out = sage(data.x, data.edge_index)
        F.cross_entropy(out[data.train_mask], data.y[data.train_mask]).backward()
        opt.step()
        sage.eval()
        with torch.no_grad():
            pred = out.argmax(dim=1)
            val_acc  = (pred[data.val_mask]  == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if val_acc > best_val:
            best_val, best_test = val_acc, test_acc

    print(f"GraphSAGE on Cora: Test accuracy = {best_test:.4f}")
    print(f"  (GCN typically ~81%, GraphSAGE typically ~81-82%)")
else:
    print("GraphSAGE on Cora typically achieves ~81-82% test accuracy.")
    print("Key advantage over GCN: inductive — works on unseen nodes at test time.")

In [ ]:
# ── GraphSAGE from scratch: mean aggregator ────────────────────────────────────
G = nx.karate_club_graph()
n = G.number_of_nodes()
adj = {node: list(G.neighbors(node)) for node in G.nodes()}
y_kc = torch.LongTensor([0 if G.nodes[i]["club"]=="Mr. Hi" else 1 for i in range(n)])

class SAGEMeanLayer(nn.Module):
    """GraphSAGE mean aggregator layer."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim * 2, out_dim)

    def forward(self, H, adj):
        """H: (N, D), adj: dict node -> [neighbours]"""
        agg = torch.zeros(len(H), H.shape[1])
        for i in range(len(H)):
            nbrs = adj[i]
            if nbrs:
                agg[i] = H[list(nbrs)].mean(dim=0)
        combined = torch.cat([H, agg], dim=1)
        return F.relu(self.W(combined))

class SAGENet(nn.Module):
    def __init__(self, in_d, hid_d, out_d):
        super().__init__()
        self.l1 = SAGEMeanLayer(in_d, hid_d)
        self.l2 = SAGEMeanLayer(hid_d, out_d)
    def forward(self, X, adj):
        return self.l2(self.l1(X, adj), adj)

sage_manual = SAGENet(n, 16, 2)
opt = torch.optim.Adam(sage_manual.parameters(), lr=0.01)
X_kc = torch.eye(n)
for _ in range(300):
    out = sage_manual(X_kc, adj)
    loss = F.cross_entropy(out, y_kc)
    opt.zero_grad(); loss.backward(); opt.step()

sage_manual.eval()
with torch.no_grad():
    preds = sage_manual(X_kc, adj).argmax(dim=1)
acc = (preds == y_kc).float().mean()
print(f"Manual GraphSAGE (mean) on Karate Club: accuracy={acc:.3f}")

In [ ]:
pos = nx.spring_layout(G, seed=42)
colours = ["steelblue" if p==0 else "tomato" for p in preds.numpy()]
plt.figure(figsize=(7, 5))
nx.draw(G, pos=pos, node_color=colours, with_labels=True, font_size=7, node_size=200, edge_color="grey", alpha=0.8)
plt.title(f"GraphSAGE (mean) predictions on Karate Club  (acc={acc:.2f})")
plt.tight_layout()
plt.savefig(f"{base}/05_graphsage.png", dpi=120)
plt.show()